In [1]:
# !pip install pysr

In [2]:
import numpy as np
import pandas as pd
import pickle
from pysr import PySRRegressor

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [3]:


model = PySRRegressor(
    maxsize=10,
    niterations=1000,
    binary_operators=["*", "+", "-", "/"],#,"^"
    # unary_operators=[
    #     "cos",
    #     "exp",
    #     "sin",
    #     "inv(x) = 1/x",
    #     "sqrt",
    #     "log",

    #     # ^ Custom operator (julia syntax)
    # ],
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    # ^ Define operator for SymPy as well
    elementwise_loss="loss(prediction, target) = abs(prediction - target)",
    # ^ Custom loss function (julia syntax)
)

In [ ]:
df = pd.read_excel("./Interpretability_GNN-Arithmetics_IndustrialSDM.xlsx","WR_MLP")

In [5]:
df.shape

(800, 8)

In [5]:
input_Train=df[['inp1','inp2','inp3','inp4','inp5','inp6','inp7','inp8']]#,'inp9','inp10','inp11']]
output_Train=df[['out1','out2']]

In [6]:
model.fit(input_Train,output_Train)

/Users/ehsan/anaconda3/envs/RL_env/lib/python3.8/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 5.950e+05
Progress: 3994 / 62000 total iterations (6.442%)
════════════════════════════════════════════════════════════════════════════════════════════════════
Best equations for output 1
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.184e+02  0.000e+00  y₀ = inp8
3           6.479e+00  1.947e+00  y₀ = inp8 * 0.93056
5           5.323e+00  9.826e-02  y₀ = (inp8 + -4.6429) * 0.9313
7           5.077e-01  1.175e+00  y₀ = (inp7 * -0.059206) + (inp8 * 0.93251)
9           4.634e-01  4.563e-02  y₀ = (inp7 * -0.059206) + ((-0.087379 + inp8) * 0.93251)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Best equations for output 2
──────────────────────────────────────────────────────────────────────────

[ Info: Final populations:
[ Info: Output 1:
[ Info: Output 2:
[ Info: Results saved to:


PySRRegressor.equations_ = [
[
	   pick     score                                           equation  \
	0        0.000000                                               inp8   
	1        1.948945                                   inp8 * 0.9307021   
	2        0.098574                      (inp8 - 4.800173) * 0.9314281   
	3  >>>>  1.569542         (inp8 - (inp7 * 0.063319616)) * 0.93241364   
	4        0.002827  ((inp8 - (inp7 * 0.063319616)) * 0.93241364) +...   
	
	         loss  complexity  
	0  318.403260           1  
	1    6.458705           3  
	2    5.303038           5  
	3    0.229741           7  
	4    0.228445           9  
], [
	   pick     score                                           equation  \
	0        0.000000                                         -157.52052   
	1        2.859464                                inp8 * -0.043436557   
	2        0.085189                  (inp8 + -6.706907) * -0.043486208   
	3        0.801172         ((inp7 * 0.09111242) - inp8) * 0.043534033   
	4  >>>>  0.330465  (((inp7 * 0.09111242) - inp8) * 0.043534033) -...   
	
	         loss  complexity  
	0  134.743840           1  
	1    0.442395           3  
	2    0.373092           5  
	3    0.075149           7  
	4    0.038805           9  
]]

In [13]:
sp.simplify(model.get_best()[1]['equation'])

(-inp10*(inp1 + inp6) - 3.17451049502808*inp10 - 8.9865103898284)/(0.76987356*inp10 + 2.1793838)

In [7]:
model.sympy()

[(-0.063319616*inp7 + inp8)*0.93241364,
 (inp7*0.09111242 - inp8)*0.043534033 - 1*0.043534033]

In [22]:
model.equations[0]

/Users/ehsan/anaconda3/envs/RL_env/lib/python3.8/site-packages/pysr/sr.py:1325: FutureWarning: PySRRegressor.equations is now deprecated. Please use PySRRegressor.equations_ instead.
  warnings.warn(


,complexity,loss,equation,score,sympy_format,lambda_format
0,1,5.905920,-14.162087,0.000000,-14.1620870000000,PySRFunction(X=>-14.1620870000000)
1,3,0.863102,inp9 * -7.982334,0.961589,inp9*(-7.982334),PySRFunction(X=>inp9*(-7.982334))
2,5,0.371821,(-9.505519 * inp9) - -3.5901272,0.421060,-9.505519*inp9 - 1*(-3.5901272),PySRFunction(X=>-9.505519*inp9 - 1*(-3.5901272))
3,7,0.249603,(inp4 * 0.00027241503) + (inp1 * -0.40909752),0.199270,inp1*(-0.40909752) + inp4*0.00027241503,PySRFunction(X=>inp1*(-0.40909752) + inp4*0.00...
4,9,0.146802,(inp1 * -0.40946624) - ((inp4 * -0.0003901399)...,0.265394,inp1*(-0.40946624) - (inp4*(-0.0003901399) + i...,PySRFunction(X=>inp1*(-0.40946624) - (inp4*(-0...
5,11,0.069229,((inp4 * 0.00053089263) - (inp6 / 0.44595578))...,0.375831,inp4*0.00053089263 - inp6/0.44595578 + inp9*(-...,PySRFunction(X=>inp4*0.00053089263 - inp6/0.44...
6,13,0.042996,(0.42006388 - inp6) + ((inp9 * -8.3117695) - (...,0.238157,-inp6 + inp9*(-8.3117695) - (inp4*(-0.00047225...,PySRFunction(X=>-inp6 + inp9*(-8.3117695) - (i...
7,15,0.041474,((inp9 * -8.311638) - ((inp6 / 0.95084196) + (...,0.018022,-inp6 + inp9*(-8.311638) - (-0.0004783105*inp4...,PySRFunction(X=>-inp6 + inp9*(-8.311638) - (-0...
8,17,0.041474,((inp9 * -8.311638) - (((inp4 + -0.14530328) *...,0.000004,inp9*(-8.311638) - (inp6 + inp6/0.95084196 + (...,PySRFunction(X=>inp9*(-8.311638) - (inp6 + inp...
9,19,0.041323,(((inp9 * -8.311637) - ((inp4 * -0.00047830303...,0.001822,inp9*(-8.311637) - (inp4*(-0.00047830303) + in...,PySRFunction(X=>inp9*(-8.311637) - (inp4*(-0.0...


In [9]:
model = PySRRegressor.from_file(run_directory="/Users/ehsan/Downloads/outputs/IS_variable/")

Attempting to load model from /Users/ehsan/Downloads/outputs/IS_variable/checkpoint.pkl...


In [10]:
model

PySRRegressor.equations_ = [
[
	   pick     score                                           equation  \
	0        0.000000                                          0.8343648   
	1        0.715337                                   inp2 / 1.7698324   
	2        0.094742                   (inp2 / 1.6116197) + -0.25993624   
	3        0.470132                 (inp2 * (inp2 / inp1)) * 4.8260837   
	4  >>>>  0.129319  (((inp2 * inp2) / inp1) * 5.0254154) - 0.06779184   
	5        0.109403  ((inp2 * (inp2 / inp1)) * 4.9681907) - (inp3 *...   
	6        0.034843  (inp2 * ((inp2 * 5.565588) / (inp2 + inp1))) -...   
	7        0.001677  (((((inp3 * ((inp3 / inp2) / -2.0702364)) + in...   
	
	       loss  complexity  
	0  0.905824           1  
	1  0.216626           3  
	2  0.179233           5  
	3  0.069995           7  
	4  0.054043           9  
	5  0.043423          11  
	6  0.040500          13  
	7  0.040094          19  
], [
	   pick     score                                           equation  \
	0        0.000000                                          -11.83177   
	1        0.155472                                 inp4 * -0.05211988   
	2        0.603681                          (inp3 + inp2) * -3.112561   
	3        1.974103            (inp2 * -2.589708) + (inp3 * -4.323892)   
	4        0.017897  ((inp2 / -0.38655207) + -0.012962911) + (inp3 ...   
	5  >>>>  0.201917  (inp3 * -4.358135) + (((inp2 / (inp3 - inp1)) ...   
	6        0.000723  (((inp2 / (((inp3 + inp3) - inp1) * 1.3894197)...   
	7        0.044400  (inp3 * -4.349543) + (((inp2 / (inp3 - (((inp1...   
	
	       loss  complexity  
	0  4.723486           1  
	1  3.461160           3  
	2  1.034835           5  
	3  0.019961           7  
	4  0.019259           9  
	5  0.008588          13  
	6  0.008563          17  
	7  0.007835          19  
]]